In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [2]:
full_charging_df = pd.read_feather('../data/full_charging_df.feather')

In [3]:
full_charging_df.head(1)

,id,connectionTime,disconnectTime,doneChargingTime,kWhDelivered,sessionID,siteID,spaceID,stationID,timezone,...,connectionMonth,connectionWeekdayName,connectionYear,isWeekend,connectionHour,disconnectHour,doneChargingHour,loading_duration,connected_duration,ratio_loading_to_connected
0,5e23b149f9af8b5fe4b973cf,2020-01-02 05:08:54-08:00,2020-01-02 11:11:15-08:00,2020-01-02 09:31:35-08:00,25.016,1_1_179_810_2020-01-02 13:08:53.870034,1,AG-3F30,1-1-179-810,America/Los_Angeles,...,1,Thursday,2020,False,5,11,9.0,262.683333,362.35,0.724944


In [4]:
# Convert to UTC because i somehow get errors else
full_charging_df['connectionTime_utc'] = full_charging_df['connectionTime'].dt.tz_convert('UTC')
full_charging_df['disconnectTime_utc'] = full_charging_df['disconnectTime'].dt.tz_convert('UTC')

full_charging_df['all_hours'] = full_charging_df.apply(
    lambda row: pd.date_range(
        start=row['connectionTime_utc'].floor('h'),
        end=row['disconnectTime_utc'].ceil('h'),
        freq='h',
        tz='UTC'
    ),
    axis=1
)

hourly_df = full_charging_df.explode('all_hours')
hourly_df['local_hour'] = hourly_df['all_hours'].dt.tz_convert('America/Los_Angeles')

In [5]:
len(full_charging_df["all_hours"])

65037

## Counting the number of connections per hour

In [6]:
counting_df = (
    hourly_df
    .groupby('local_hour')
    .agg(session_count=('sessionID', 'nunique'))
    .reset_index()
)

counting_df

,local_hour,session_count
0,2018-04-25 04:00:00-07:00,1
1,2018-04-25 05:00:00-07:00,1
2,2018-04-25 06:00:00-07:00,3
3,2018-04-25 07:00:00-07:00,8
4,2018-04-25 08:00:00-07:00,22
...,...,...
24366,2021-09-14 04:00:00-07:00,1
24367,2021-09-14 05:00:00-07:00,1
24368,2021-09-14 06:00:00-07:00,1
24369,2021-09-14 07:00:00-07:00,1


In [7]:
def get_count(time: pd.Timestamp):
    try:
        return counting_df.loc[counting_df["local_hour"] == time]["session_count"].iloc[0]
    except:
        return 0

In [8]:
specific_time = pd.Timestamp('2019-01-18 22:00:00-08:00')
get_count(specific_time)

np.int64(9)

## Mean delivered kWh per hour

In [9]:
hourly_df['hour_start'] = hourly_df['all_hours']
hourly_df['hour_end']   = hourly_df['hour_start'] + pd.Timedelta(hours=1)
hourly_df['overlap_start'] = hourly_df[['connectionTime_utc','hour_start']].max(axis=1)
hourly_df['overlap_end']   = hourly_df[['disconnectTime_utc','hour_end']].min(axis=1)

hourly_df['minutes_in_hour'] = (hourly_df['overlap_end'] - hourly_df['overlap_start']).dt.total_seconds() / 60.0
hourly_df['fraction_of_session'] = hourly_df['minutes_in_hour'] / hourly_df['loading_duration']
hourly_df['kWh_in_this_hour'] = hourly_df['kWhDelivered'] * hourly_df['fraction_of_session']
kpi_df = hourly_df.groupby('local_hour').agg(
    total_kWh=('kWh_in_this_hour', 'sum'),
    session_count=('sessionID', 'nunique'),
)

kpi_df = kpi_df.reset_index()

In [10]:
kpi_df.head()

,local_hour,total_kWh,session_count
0,2018-04-25 04:00:00-07:00,3.094930,1
1,2018-04-25 05:00:00-07:00,3.575657,1
2,2018-04-25 06:00:00-07:00,3.161296,3
3,2018-04-25 07:00:00-07:00,10.509661,8
4,2018-04-25 08:00:00-07:00,41.767757,22


In [11]:
kpi_df.to_feather('../data/kpi_df.feather')